# Adapter Pattern — Exercises

Run `example.ipynb` first to load the base classes conceptually (each exercise below is self-contained and re-declares what it needs).

## Exercise 1 (basic application) — Adapt a second legacy source

**Goal**: A new legacy class `CSVDataSource` exposes `fetch_csv() -> str`, returning `"id,name\n42,Ada Lovelace"`. Write `CSVToJSONAdapter` implementing `JSONDataSource` that parses this into `{"id": "42", "name": "Ada Lovelace"}`.

**Acceptance criteria**:
- `CSVToJSONAdapter` implements the `JSONDataSource` interface (has `fetch_json`).
- It does not modify `CSVDataSource`.
- `print_user_report(CSVToJSONAdapter(CSVDataSource()))` prints `User #42: Ada Lovelace`.

In [1]:
from abc import ABC, abstractmethod


class JSONDataSource(ABC):
    @abstractmethod
    def fetch_json(self) -> dict:
        raise NotImplementedError


class CSVDataSource:
    def fetch_csv(self) -> str:
        return "id,name\n42,Ada Lovelace"


def print_user_report(source: JSONDataSource) -> None:
    data = source.fetch_json()
    print(f"User #{data['id']}: {data['name']}")


class CSVToJSONAdapter(JSONDataSource):
    def __init__(self, csv_source: CSVDataSource):
        self._csv_source = csv_source

    def fetch_json(self) -> dict:
        csv_data = self._csv_source.fetch_csv()
        lines = csv_data.split("\n")
        headers = lines[0].split(",")
        values = lines[1].split(",")
        return {headers[i]: values[i] for i in range(len(headers))}

        


print_user_report(CSVToJSONAdapter(CSVDataSource()))


User #42: Ada Lovelace


## Exercise 2 (adaptation) — One adapter, multiple adaptees

**Goal**: Instead of writing a separate adapter class per format, design a single `DictAdapter` that accepts *any* object with a `.fetch_raw() -> str` method plus a `parser` function (e.g. `json.loads`, a custom XML parser, a custom CSV parser) injected at construction time. It should implement `JSONDataSource`.

**Acceptance criteria**:
- `DictAdapter(some_source, parser=some_parser_fn)` implements `fetch_json()` by calling `some_source.fetch_raw()` then `some_parser_fn(...)`.
- Show it working with at least two different adaptee/parser combinations.
- Explain in a markdown cell: does this still count as the Adapter pattern, or has it drifted toward Strategy? Justify your answer.

In [2]:
import json


class RawJSONSource:
    def fetch_raw(self) -> str:
        return '{"id": "42", "name": "Ada Lovelace"}'


class RawCSVSource:
    def fetch_raw(self) -> str:
        return "id,name\n7,Grace Hopper"


def parse_csv(data: str) -> dict:
    headers, values = data.splitlines()
    return dict(zip(headers.split(","), values.split(",")))


class DictAdapter(JSONDataSource):
    def __init__(self, source, parser):
        self._source = source
        self._parser = parser

    def fetch_json(self) -> dict:
        raw_data = self._source.fetch_raw()
        return self._parser(raw_data)


json_adapter = DictAdapter(RawJSONSource(), json.loads)
csv_adapter = DictAdapter(RawCSVSource(), parse_csv)

print_user_report(json_adapter)
print_user_report(csv_adapter)


User #42: Ada Lovelace
User #7: Grace Hopper


### Adapter or Strategy?

This is still an Adapter because `DictAdapter` converts the adaptees' `fetch_raw()` interface into the `JSONDataSource.fetch_json()` interface expected by the client. It also uses composition and leaves the adaptees unchanged. However, the injected `parser` is a Strategy: it encapsulates a variable parsing algorithm that can be selected at runtime. The design therefore combines an Adapter with a Strategy, rather than being only one pattern.

## Exercise 3 (critique) — Class Adapter vs Object Adapter

**Goal**: Rewrite the `XMLToJSONAdapter` from `example.ipynb` as a **Class Adapter** using multiple inheritance (`class XMLToJSONAdapter(JSONDataSource, XMLDataSource)`), instead of composition.

**Acceptance criteria**:
- The class adapter version behaves identically to the object adapter version for the demo call.
- Write 2-3 sentences comparing the two versions: which one would break if `XMLDataSource` later required constructor arguments, and why the object-adapter version is generally preferred in Python?

In [3]:
import xml.etree.ElementTree as ET


class XMLDataSource:
    def fetch_xml(self) -> str:
        return "<user><id>42</id><name>Ada Lovelace</name></user>"


class XMLToJSONAdapter(JSONDataSource, XMLDataSource):
    def fetch_json(self) -> dict:
        xml_string = self.fetch_xml()
        root = ET.fromstring(xml_string)
        return {child.tag: child.text for child in root}


adapter = XMLToJSONAdapter()
print_user_report(adapter)


User #42: Ada Lovelace


### Class Adapter vs Object Adapter

A class adapter uses multiple inheritance, so `XMLToJSONAdapter` is both a `JSONDataSource` and an `XMLDataSource`. The adapter can call the inherited `fetch_xml()` method directly, then translate its result in `fetch_json()`.

If `XMLDataSource` later requires constructor arguments, the class-adapter demo must change to pass those arguments when creating `XMLToJSONAdapter`, while the object adapter can receive an already-configured `XMLDataSource` instance. The object-adapter approach is generally preferred in Python because composition is more flexible and avoids tighter coupling through multiple inheritance.